# fsm-llm-venser on Colab

Installs the package from GitHub, gets the example machine and runs a scenario.
The code is installed from the release tag set in TAG below.

In [ ]:
REPO = "WhiteEnderman14/fsm-llm-venser"
TAG = "v0.1.0"
!pip install -q "fsm-llm-venser[ollama,gemini,claude] @ git+https://github.com/{REPO}@{TAG}"
!git clone -q --depth 1 --branch {TAG} https://github.com/{REPO}.git
%cd fsm-llm-venser
!apt-get -qq install graphviz   # to draw the machine

## Model access
Use **one** of the three cells. Ollama with `llama3.2` is the main option; Gemini and Claude are untested.

In [ ]:
# Option 1: Ollama running inside this Colab session (use a GPU runtime). Main model: llama3.2.
import subprocess
import time
import urllib.request

!apt-get -qq install zstd     # the Ollama installer needs it
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"])
for _ in range(60):           # wait until the server answers
    try:
        urllib.request.urlopen("http://127.0.0.1:11434")
        break
    except OSError:
        time.sleep(1)
!ollama pull llama3.2
PROVIDER, MODEL = "ollama", None

In [ ]:
# Option 2: Gemini (untested). Store the key as a Colab Secret named GOOGLE_API_KEY.
import os

from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
PROVIDER, MODEL = "gemini", None

In [ ]:
# Option 3: Claude (untested). Store the key as a Colab Secret named ANTHROPIC_API_KEY.
import os

from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
PROVIDER, MODEL = "claude", None

## Check and run

In [ ]:
EXAMPLE = "examples/narrative_medicine_interview"
!fsm-llm-venser check-llm --provider {PROVIDER}
!fsm-llm-venser validate {EXAMPLE}/machine.yaml --plugin {EXAMPLE}/plugin.py

In [ ]:
!fsm-llm-venser run {EXAMPLE}/machine.yaml --plugin {EXAMPLE}/plugin.py \
    --scenario {EXAMPLE}/scenarios/a_happy_path.yaml --provider {PROVIDER} --draw path.svg

## The path of the run
The arrows the run went through are drawn in red.

In [ ]:
from IPython.display import SVG

SVG("path.svg")